<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI in Asset Management
## Appendix C · scikit-learn Cheat Sheet

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by GPT 5.1<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh

## Notebook Goals

This notebook collects small, runnable scikit-learn recipes for regression,
classification, pipelines, and hyperparameter search.

### Getting Help

- Appendix C is the main cheat sheet; this notebook mirrors it in code.
- Chapter 10 provides more context for how these models feed portfolios.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

plt.style.use("seaborn-v0_8")
plt.rcParams.update({"font.family": "serif", "figure.dpi": 300})

DATA_PATH = Path("../data/pyaiam_eod.csv")
if not DATA_PATH.exists():
    DATA_PATH = "https://hilpisch.com/pyaiam_eod.csv"

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import mean_squared_error, roc_auc_score

### Data Loading
We reuse the equity panel for simple regression and classification demos.

In [ ]:
prices = pd.read_csv(DATA_PATH,
parse_dates=['Date']).set_index('Date').sort_index().ffill()

## 1. Regression Pipeline Example

We fit a ridge regression to predict next-day AAPL returns from momentum and volatility.

In [ ]:
log_rets = np.log(prices / prices.shift(1)).dropna()
X = pd.DataFrame(
    {
        'momentum': prices['AAPL'].pct_change(20),
        'volatility': log_rets['AAPL'].rolling(20).std(),
    }
).dropna()
y = log_rets['AAPL'].shift(-1).reindex(X.index).dropna()
X = X.loc[y.index]
pipe = Pipeline([
    ('scale', StandardScaler()),
    ('model', Ridge(alpha=10.0)),
])
X_train, X_test = X.iloc[:-252], X.iloc[-252:]
y_train, y_test = y.iloc[:-252], y.iloc[-252:]
pipe.fit(X_train, y_train)
float(mean_squared_error(y_test, pipe.predict(X_test)))

## 2. Time-Series Aware Grid Search

We wrap a pipeline in GridSearchCV using TimeSeriesSplit to avoid leakage.

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)
param_grid = {'model__alpha': [1.0, 10.0, 100.0]}
search = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=tscv,
    scoring='neg_mean_squared_error',
)
search.fit(X, y)
search.best_params_

## 3. Classification with Logistic Regression

We change the target to an up/down label and evaluate AUC.

In [ ]:
y_cls = (y > 0).astype(int)
logit = Pipeline([
    ('scale', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000)),
])
logit.fit(X_train, y_cls.iloc[:-252])
probs = logit.predict_proba(X_test)[:, 1]
float(roc_auc_score(y_cls.iloc[-252:], probs))

## 4. Exercises

1. Add an ElasticNet model to the grid search.
2. Swap StandardScaler for RobustScaler and compare metrics.
3. Implement a reusable helper that builds a GridSearchCV object from a model and parameter dictionary.

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">